<a href="https://colab.research.google.com/github/Foyin-Tony/Student_grad_risk_tracker/blob/main/Grad_track.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)  # reproducibility - same "random" numbers every run

N = 2000  # number of students

# Entry mode: UTME (direct JAMB entry) vs Direct Entry (ND/HND holders, like you)
entry_mode = np.random.choice(["UTME", "Direct Entry"], size=N, p=[0.75, 0.25])

# Attendance rate: proportion of classes attended (0 to 1)
attendance_rate = np.clip(np.random.beta(a=6, b=2, size=N), 0, 1)

# Study hours per week (self-reported average across semesters)
study_hours = np.clip(np.random.normal(loc=15, scale=6, size=N), 0, 40)

# 100-level (or entry-level) CGPA - the students' starting point
entry_cgpa = np.clip(np.random.normal(loc=3.6, scale=0.6, size=N), 1.0, 5.0)

# Course load per semester (number of courses)
course_load = np.random.choice([5, 6, 7, 8], size=N, p=[0.15, 0.35, 0.35, 0.15])

# Financial stress indicator (0 = low, 1 = high)
financial_stress = np.random.binomial(1, 0.35, size=N)

# Part-time work (hours per week) - trades off against study time
part_time_work_hours = np.where(
    financial_stress == 1,
    np.clip(np.random.normal(12, 5, size=N), 0, 30),
    np.clip(np.random.normal(2, 2, size=N), 0, 10)
)

df = pd.DataFrame({
    "entry_mode": entry_mode,
    "attendance_rate": attendance_rate,
    "study_hours_per_week": study_hours,
    "entry_cgpa": entry_cgpa,
    "course_load": course_load,
    "financial_stress": financial_stress,
    "part_time_work_hours": part_time_work_hours,
})

print(df.head())
print(df.describe())

df.to_csv("predictors_only.csv", index=False)
print("Saved predictors_only.csv")


     entry_mode  attendance_rate  study_hours_per_week  entry_cgpa  \
0          UTME         0.825017             22.537185    3.626226   
1  Direct Entry         0.907388             22.126280    4.333685   
2          UTME         0.865042             10.104543    3.259315   
3          UTME         0.345898              8.077762    3.907526   
4          UTME         0.706119             17.979808    3.774053   

   course_load  financial_stress  part_time_work_hours  
0            7                 0              4.456655  
1            6                 1             14.906865  
2            6                 0              0.000000  
3            7                 0              3.390538  
4            7                 0              2.728340  
       attendance_rate  study_hours_per_week   entry_cgpa  course_load  \
count      2000.000000           2000.000000  2000.000000  2000.000000   
mean          0.745556             14.905282     3.596974     6.532000   
std           0

In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)
df = pd.read_csv("predictors_only.csv")

def standardize(x):
    return (x - x.mean()) / x.std()

z_attendance = standardize(df["attendance_rate"])
z_entry_cgpa = standardize(df["entry_cgpa"])
z_study_hours = standardize(df["study_hours_per_week"])
z_fin_stress = df["financial_stress"]
z_entry_mode = (df["entry_mode"] == "Direct Entry").astype(int)

# TRUE coefficients (our "ground truth")
b0 = 0.9
b_attendance = 1.1
b_entry_cgpa = 0.8
b_study_hours = 0.4
b_fin_stress = -0.5
b_entry_mode = -0.2

log_odds = (
    b0
    + b_attendance * z_attendance
    + b_entry_cgpa * z_entry_cgpa
    + b_study_hours * z_study_hours
    + b_fin_stress * z_fin_stress
    + b_entry_mode * z_entry_mode
)

p_graduate = 1 / (1 + np.exp(-log_odds))
graduated = np.random.binomial(1, p_graduate)

df["true_log_odds"] = log_odds
df["true_p_graduate"] = p_graduate
df["graduated"] = graduated

print(df["graduated"].value_counts(normalize=True))
print(df.groupby(pd.qcut(df["attendance_rate"], 4))["graduated"].mean())

df.to_csv("full_dataset_v1.csv", index=False)
print("Saved full_dataset_v1.csv")

graduated
1    0.6265
0    0.3735
Name: proportion, dtype: float64
attendance_rate
(0.205, 0.653]    0.358
(0.653, 0.76]     0.622
(0.76, 0.858]     0.708
(0.858, 0.997]    0.818
Name: graduated, dtype: float64
Saved full_dataset_v1.csv


/tmp/ipykernel_2574/2746251717.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby(pd.qcut(df["attendance_rate"], 4))["graduated"].mean())


In [3]:
import numpy as np
import pandas as pd

np.random.seed(303)  # DIFFERENT seed from earlier cells - matters, explained below
df = pd.read_csv("full_dataset_v1.csv")

def standardize(x):
    return (x - x.mean()) / x.std()

z_attendance = standardize(df["attendance_rate"])
z_study_hours = standardize(df["study_hours_per_week"])
z_fin_stress = df["financial_stress"]

effect_attendance = 0.35
effect_study_hours = 0.15
effect_fin_stress = -0.20

noise = np.random.normal(0, 0.25, size=len(df))

final_cgpa = (
    df["entry_cgpa"]
    + effect_attendance * z_attendance
    + effect_study_hours * z_study_hours
    + effect_fin_stress * z_fin_stress
    + noise
)
final_cgpa = np.clip(final_cgpa, 1.0, 5.0)
df["final_cgpa"] = final_cgpa

def classify(cgpa):
    if cgpa >= 4.50:
        return "First Class"
    elif cgpa >= 3.50:
        return "Second Class Upper"
    elif cgpa >= 2.40:
        return "Second Class Lower"
    else:
        return "Third Class"

df["class_of_degree"] = df["final_cgpa"].apply(classify)
df.loc[df["graduated"] == 0, "class_of_degree"] = "N/A - Did Not Graduate"

print(df["class_of_degree"].value_counts())
print(pd.crosstab(df["graduated"], df["class_of_degree"]))

df.to_csv("full_dataset_v2.csv", index=False)
print("Saved full_dataset_v2.csv")

class_of_degree
N/A - Did Not Graduate    747
Second Class Upper        663
Second Class Lower        404
First Class               166
Third Class                20
Name: count, dtype: int64
class_of_degree  First Class  N/A - Did Not Graduate  Second Class Lower  \
graduated                                                                  
0                          0                     747                   0   
1                        166                       0                 404   

class_of_degree  Second Class Upper  Third Class  
graduated                                         
0                                 0            0  
1                               663           20  
Saved full_dataset_v2.csv


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score

df = pd.read_csv("full_dataset_v2.csv")

def standardize(x):
    return (x - x.mean()) / x.std()

df["z_attendance"] = standardize(df["attendance_rate"])
df["z_entry_cgpa"] = standardize(df["entry_cgpa"])
df["z_study_hours"] = standardize(df["study_hours_per_week"])
df["z_entry_mode"] = (df["entry_mode"] == "Direct Entry").astype(int)

feature_cols = ["z_attendance", "z_entry_cgpa", "z_study_hours", "financial_stress", "z_entry_mode"]
X = df[feature_cols].values
y = df["graduated"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

model = LogisticRegression(penalty=None, max_iter=1000)
model.fit(X_train, y_train)

coefs = pd.Series(model.coef_[0], index=feature_cols)
print("\nEstimated coefficients (log-odds):")
print(coefs.round(3))

# Bootstrap for confidence intervals
n_bootstrap = 500
boot_coefs = np.zeros((n_bootstrap, len(feature_cols)))
rng = np.random.RandomState(404)
n_train = len(X_train)

for i in range(n_bootstrap):
    idx = rng.choice(n_train, size=n_train, replace=True)
    X_boot, y_boot = X_train[idx], y_train[idx]
    m = LogisticRegression(penalty=None, max_iter=1000)
    m.fit(X_boot, y_boot)
    boot_coefs[i] = m.coef_[0]

ci_lower = np.percentile(boot_coefs, 2.5, axis=0)
ci_upper = np.percentile(boot_coefs, 97.5, axis=0)
odds_ratios = np.exp(coefs)

summary_table = pd.DataFrame({
    "coefficient": coefs.values,
    "odds_ratio": odds_ratios.values,
    "OR_lower_95CI": np.exp(ci_lower),
    "OR_upper_95CI": np.exp(ci_upper),
}, index=feature_cols)
print("\nOdds Ratios with 95% Bootstrap Confidence Intervals:")
print(summary_table.round(3))

# Compare to TRUE coefficients we built in
true_coefs = {"z_attendance": 1.1, "z_entry_cgpa": 0.8, "z_study_hours": 0.4,
              "financial_stress": -0.5, "z_entry_mode": -0.2}
comparison = pd.DataFrame({
    "TRUE": pd.Series(true_coefs),
    "ESTIMATED": coefs,
})
print("\nTRUE vs ESTIMATED:")
print(comparison.round(3))

# ROC/AUC on test set
y_pred_prob = model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)
print(f"\nTest-set AUC: {auc:.3f}")

df.to_csv("full_dataset_v3_with_features.csv", index=False)
pd.DataFrame(X_test, columns=feature_cols).to_csv("X_test.csv", index=False)
pd.Series(y_test).to_csv("y_test.csv", index=False)

Train size: 1600, Test size: 400

Estimated coefficients (log-odds):
z_attendance        1.056
z_entry_cgpa        0.936
z_study_hours       0.405
financial_stress   -0.594
z_entry_mode       -1.607
dtype: float64

Odds Ratios with 95% Bootstrap Confidence Intervals:
                  coefficient  odds_ratio  OR_lower_95CI  OR_upper_95CI
z_attendance            1.056       2.875          2.520          3.370
z_entry_cgpa            0.936       2.550          2.215          2.970
z_study_hours           0.405       1.499          1.325          1.707
financial_stress       -0.594       0.552          0.438          0.701
z_entry_mode           -1.607       0.200          0.130          0.289

TRUE vs ESTIMATED:
                  TRUE  ESTIMATED
z_attendance       1.1      1.056
z_entry_cgpa       0.8      0.936
z_study_hours      0.4      0.405
financial_stress  -0.5     -0.594
z_entry_mode      -0.2     -1.607

Test-set AUC: 0.847


In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score

df = pd.read_csv("full_dataset_v3_with_features.csv")

feature_cols = ["z_attendance", "z_entry_cgpa", "z_study_hours", "financial_stress", "z_entry_mode"]
X = df[feature_cols].values
y = df["graduated"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=5, random_state=505)
rf.fit(X_train, y_train)
rf_pred_prob = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_pred_prob)

rf_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Random Forest: Variable Importance")
print(rf_importance.round(3))
print(f"\nRandom Forest Test AUC: {rf_auc:.3f}")

# Gradient Boosting (our XGBoost substitute)
gbm = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=606)
gbm.fit(X_train, y_train)
gbm_pred_prob = gbm.predict_proba(X_test)[:, 1]
gbm_auc = roc_auc_score(y_test, gbm_pred_prob)

gbm_importance = pd.Series(gbm.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nGradient Boosting: Variable Importance")
print(gbm_importance.round(3))
print(f"\nGradient Boosting Test AUC: {gbm_auc:.3f}")

# Need the logistic regression model object from Cell 4 still in memory
logit_pred_prob = model.predict_proba(X_test)[:, 1]
logit_auc = roc_auc_score(y_test, logit_pred_prob)

print("\n=== MODEL COMPARISON: Test AUC ===")
print(f"Logistic Regression : {logit_auc:.3f}")
print(f"Random Forest        : {rf_auc:.3f}")
print(f"Gradient Boosting     : {gbm_auc:.3f}")

np.save("rf_pred_prob.npy", rf_pred_prob)
np.save("gbm_pred_prob.npy", gbm_pred_prob)
np.save("logit_pred_prob.npy", logit_pred_prob)
np.save("y_test_array.npy", y_test)

Random Forest: Variable Importance
z_attendance        0.394
z_entry_cgpa        0.275
z_entry_mode        0.220
z_study_hours       0.093
financial_stress    0.018
dtype: float64

Random Forest Test AUC: 0.933

Gradient Boosting: Variable Importance
z_attendance        0.304
z_entry_cgpa        0.297
z_entry_mode        0.292
z_study_hours       0.093
financial_stress    0.014
dtype: float64

Gradient Boosting Test AUC: 0.917

=== MODEL COMPARISON: Test AUC ===
Logistic Regression : 0.847
Random Forest        : 0.933
Gradient Boosting     : 0.917


In [6]:
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve

y_test = np.load("y_test_array.npy")
logit_pred = np.load("logit_pred_prob.npy")
rf_pred = np.load("rf_pred_prob.npy")
gbm_pred = np.load("gbm_pred_prob.npy")

for name, preds in [("Logistic Regression", logit_pred), ("Random Forest", rf_pred), ("Gradient Boosting", gbm_pred)]:
    prob_true, prob_pred = calibration_curve(y_test, preds, n_bins=10, strategy="quantile")
    print(f"=== {name} calibration (10 bins) ===")
    calib_df = pd.DataFrame({
        "avg_predicted_prob": prob_pred.round(3),
        "actual_observed_rate": prob_true.round(3),
        "difference": (prob_true - prob_pred).round(3),
    })
    print(calib_df)
    print(f"Mean absolute calibration error: {np.mean(np.abs(prob_true - prob_pred)):.4f}\n")

np.save("calib_logit.npy", np.array(calibration_curve(y_test, logit_pred, n_bins=10, strategy="quantile")))
np.save("calib_rf.npy", np.array(calibration_curve(y_test, rf_pred, n_bins=10, strategy="quantile")))
np.save("calib_gbm.npy", np.array(calibration_curve(y_test, gbm_pred, n_bins=10, strategy="quantile")))

=== Logistic Regression calibration (10 bins) ===
   avg_predicted_prob  actual_observed_rate  difference
0               0.137                 0.625       0.488
1               0.328                 0.100      -0.228
2               0.431                 0.000      -0.431
3               0.518                 0.275      -0.243
4               0.639                 0.650       0.011
5               0.727                 0.825       0.098
6               0.792                 0.925       0.133
7               0.857                 0.900       0.043
8               0.925                 0.975       0.050
9               0.970                 1.000       0.030
Mean absolute calibration error: 0.1755

=== Random Forest calibration (10 bins) ===
   avg_predicted_prob  actual_observed_rate  difference
0               0.226                 0.025      -0.201
1               0.352                 0.150      -0.202
2               0.427                 0.275      -0.152
3               0.485    

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("full_dataset_v3_with_features.csv")
feature_cols = ["z_attendance", "z_entry_cgpa", "z_study_hours", "financial_stress", "z_entry_mode"]
X = df[feature_cols].values
y = df["graduated"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_b = np.column_stack([np.ones(len(X_train)), X_train])
n_params = X_train_b.shape[1]

def log_likelihood(beta, X, y):
    log_odds = X @ beta
    log_lik = y * log_odds - np.log1p(np.exp(log_odds))
    return np.sum(log_lik)

def log_prior(beta, prior_sigma=2.0):
    return np.sum(-0.5 * (beta / prior_sigma) ** 2)

def log_posterior(beta, X, y):
    return log_likelihood(beta, X, y) + log_prior(beta)

np.random.seed(707)
n_iterations = 20000
burn_in = 5000
proposal_sd = 0.05

beta_current = np.zeros(n_params)
log_post_current = log_posterior(beta_current, X_train_b, y_train)

samples = np.zeros((n_iterations, n_params))
n_accepted = 0

for i in range(n_iterations):
    beta_proposed = beta_current + np.random.normal(0, proposal_sd, size=n_params)
    log_post_proposed = log_posterior(beta_proposed, X_train_b, y_train)
    log_accept_ratio = log_post_proposed - log_post_current
    if np.log(np.random.uniform()) < log_accept_ratio:
        beta_current = beta_proposed
        log_post_current = log_post_proposed
        n_accepted += 1
    samples[i] = beta_current

acceptance_rate = n_accepted / n_iterations
print(f"MCMC acceptance rate: {acceptance_rate:.2%}")

posterior_samples = samples[burn_in:]
param_names = ["intercept"] + feature_cols
posterior_mean = posterior_samples.mean(axis=0)
credible_lower = np.percentile(posterior_samples, 2.5, axis=0)
credible_upper = np.percentile(posterior_samples, 97.5, axis=0)

results = pd.DataFrame({
    "posterior_mean_coef": posterior_mean,
    "odds_ratio": np.exp(posterior_mean),
    "95%_credible_lower": np.exp(credible_lower),
    "95%_credible_upper": np.exp(credible_upper),
}, index=param_names)

print(results.round(3))
np.save("bayesian_posterior_samples.npy", posterior_samples)

MCMC acceptance rate: 44.55%
                  posterior_mean_coef  odds_ratio  95%_credible_lower  \
intercept                       1.352       3.865               3.183   
z_attendance                    1.060       2.885               2.516   
z_entry_cgpa                    0.937       2.553               2.224   
z_study_hours                   0.408       1.503               1.320   
financial_stress               -0.594       0.552               0.431   
z_entry_mode                   -1.609       0.200               0.152   

                  95%_credible_upper  
intercept                      4.624  
z_attendance                   3.326  
z_entry_cgpa                   2.954  
z_study_hours                  1.716  
financial_stress               0.711  
z_entry_mode                   0.262  
